# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset (doi:10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. 

### Dataset Source
The dataset is described by a Croissant schema available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

_To learn more about the Croissant format: https://mlcommons.org/croissant/_

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
ds = mlc.Dataset(croissant_url)

# Print main metadata (using dataset.metadata attributes, not subscript notation)
print(f"{ds.metadata.name}: {ds.metadata.description}\nPublished: {ds.metadata.datePublished}\nVersion: {ds.metadata.version}\nLicense: {ds.metadata.license}")

## 2. Data Overview
List available record sets, their `@id`s, field names, and structures in the dataset.

In [ ]:
# List available record sets and their fields by @id

from pprint import pprint

record_sets = ds.metadata.recordSets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}  Name: {rs.get('name', '<no name>')}")
    fields = rs.get('fields', [])
    print("  Fields (@id: name, type):")
    for f in fields:
        print(f"    {f['@id']}: {f.get('name', '<no name>')} ({f.get('dataType', 'unknown')})")
    print("  ---\n")

## 3. Data Extraction
Load data for each record set as a pandas DataFrame for exploration. Use `@id` fields for clarity (see previous overview for available record sets and their fields).

In [ ]:
# List all record set @id's
record_set_ids = [rs['@id'] for rs in ds.metadata.recordSets]

# Load records for each record set as a DataFrame, keyed by record set @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for: {record_set_id} ...")
    records = list(ds.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  WARNING: No records found for {record_set_id}")
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, show columns of the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Available columns for RecordSet {first_rs}:\n{dataframes[first_rs].columns.tolist()}")
    dataframes[first_rs].head()
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate typical EDA tasks -- filtering, normalizing, grouping on one record set. _All fields referenced by their `@id`._

In [ ]:
# Example EDA: Select the first record set with numeric fields
import numpy as np

# Try to find a record set with numeric fields
eda_rs_id = None
numeric_field_id = None
group_field_id = None

for rs in ds.metadata.recordSets:
    df = dataframes.get(rs['@id'], pd.DataFrame())
    if df.empty:
        continue
    # Find field with a numeric dataType
    for f in rs.get('fields', []):
        dt = f.get('dataType', '').lower()
        if any(x in dt for x in ['integer', 'float', 'number']) and f['@id'] in df.columns:
            eda_rs_id = rs['@id']
            numeric_field_id = f['@id']
            # Find a group-by-able field
            for f2 in rs.get('fields', []):
                if f2['@id'] in df.columns and f2['@id'] != numeric_field_id:
                    group_field_id = f2['@id']
                    break
            break
    if eda_rs_id:
        break

if eda_rs_id and numeric_field_id:
    print(f"Using record set: {eda_rs_id}")
    df = dataframes[eda_rs_id]
    # Drop missing values for the numeric field
    df = df.dropna(subset=[numeric_field_id])
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Choose a threshold for filtering (median for demonstration)
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} rows")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (first few rows):")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field for EDA was found in this dataset.")

## 5. Visualization
Visualize distributions or relationships of numeric variables in the dataset using matplotlib or seaborn.

In [ ]:
# Visualize the distribution of the main numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if eda_rs_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=dataframes[eda_rs_id], x=numeric_field_id, bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {eda_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field found to plot.")

## 6. Conclusion
Exploratory analysis of the FAIR² mass spectrometry dataset reveals its structure is well-described via Croissant. Numeric data columns can be filtered, normalized, and grouped for exploratory analysis. Suitable downstream tasks include protein feature analysis, sample clustering, and biomarker discovery as described in the metadata. 

_Please refer to the official publication and Croissant file for comprehensive dataset definition and field semantics._